# Retrieval analysis
Loads the raw, LLM-free retrieval results and produces summary tables and figures.

In [ ]:
from pathlib import Path
import csv, math, statistics
from collections import defaultdict
ROOT = Path('..')
RESULTS = ROOT / 'eval' / 'results'
def read_csv(path):
    with path.open(encoding='utf-8', newline='') as f: return list(csv.DictReader(f))
def mean(rows, key):
    return sum(float(r[key]) for r in rows) / max(1, len(rows))


In [ ]:
# Aggregate ablation metrics
summary = []
for path in sorted(RESULTS.glob('retrieval_*.csv')):
    if 'tuning' in path.name: continue
    rows = read_csv(path)
    summary.append({'config': rows[0]['config'], **{m: mean(rows, m) for m in ('recall_at_5','mrr','ndcg_at_5','hit_at_1')}})
summary


In [ ]:
# Per-category heatmap data; render with seaborn/matplotlib when installed
category = defaultdict(list)
for path in sorted(RESULTS.glob('retrieval_*.csv')):
    if 'tuning' in path.name: continue
    for row in read_csv(path): category[(row['config'], row['category'])].append(float(row['recall_at_5']))
heatmap = [{'config': c, 'category': g, 'recall_at_5': sum(v)/len(v)} for (c,g),v in sorted(category.items())]
heatmap


## Statistical test
For each question, compare D and H5. Use a paired bootstrap or Wilcoxon test when scipy is installed; otherwise report the paired mean difference and confidence interval from the standard library.

In [ ]:
def by_id(name):
    return {r['id']: r for r in read_csv(RESULTS / name)}
d = by_id('retrieval_D.csv'); h5 = by_id('retrieval_H5.csv')
diffs = [float(h5[k]['recall_at_5']) - float(d[k]['recall_at_5']) for k in d if k in h5]
def bootstrap_ci(values, iterations=2000, seed=7):
    import random
    rng = random.Random(seed); means = []
    for _ in range(iterations): means.append(statistics.mean(rng.choices(values, k=len(values))))
    means.sort(); return (means[int(.025*len(means))], means[int(.975*len(means))-1])
paired_summary = {'n': len(diffs), 'mean_difference': statistics.mean(diffs) if diffs else 0.0, 'stdev': statistics.stdev(diffs) if len(diffs)>1 else 0.0, 'bootstrap_95_ci': bootstrap_ci(diffs) if diffs else (0.0, 0.0)}
try:
    from scipy.stats import wilcoxon
    paired_summary['wilcoxon'] = wilcoxon(diffs).pvalue
except ImportError:
    paired_summary['wilcoxon'] = None
paired_summary


## Error analysis and case studies
Select the 20 largest D/H5 misses from the CSVs, label each failure as retrieval miss, graph miss, entity-linking issue, or out-of-scope handling, and include 2–3 representative traces comparing D, G, and H5.

In [ ]:
errors = []
for question_id, row in d.items():
    if float(row['recall_at_5']) < 1 or float(h5.get(question_id, {'recall_at_5': 0})['recall_at_5']) < 1:
        errors.append({'id': question_id, 'D': row['predicted'], 'H5': h5.get(question_id, {}).get('predicted', ''), 'label': 'to-review'})
errors[:20]
